# Continued Fractions: Best Rational Approximations

Every real number $x$ has a representation as an infinite (or finite) **continued fraction**:
$$
x = a_0 + \cfrac{1}{a_1 + \cfrac{1}{a_2 + \cfrac{1}{a_3 + \cdots}}} = [a_0; a_1, a_2, a_3, \ldots]
$$
where $a_0 = \lfloor x \rfloor$ and $a_k = \lfloor 1/(x_{k-1} - \lfloor x_{k-1} \rfloor) \rfloor$ for $k \geq 1$. The partial quotients $a_k$ are positive integers for $k \geq 1$.

## Convergents

The **$n$-th convergent** is the rational number $p_n/q_n = [a_0; a_1, \ldots, a_n]$ obtained by truncating the expansion. Convergents satisfy the three-term recurrence:
$$
p_n = a_n p_{n-1} + p_{n-2}, \qquad q_n = a_n q_{n-1} + q_{n-2},
$$
with $p_{-1} = 1, p_0 = a_0, q_{-1} = 0, q_0 = 1$.

## Best approximation property

Convergents are the **best rational approximations** to $x$: among all rationals $p/q$ with denominator $q \leq q_n$, the convergent $p_n/q_n$ is closest to $x$. Moreover:
$$
\left|x - \frac{p_n}{q_n}\right| < \frac{1}{q_n q_{n+1}} \leq \frac{1}{q_n^2}.
$$
This $O(1/q^2)$ convergence rate is optimal for generic irrational numbers (Hurwitz's theorem gives the best constant $1/\sqrt{5} q^2$).

## The golden ratio and worst-case approximation

The **golden ratio** $\phi = [1; 1, 1, 1, \ldots]$ has all partial quotients equal to $1$ — the smallest possible. Its convergents are the Fibonacci ratios $F_{n+1}/F_n$, and it is the **hardest number to approximate** by rationals (slowest convergence among all irrationals).

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

plt.rcParams['figure.dpi'] = 120

## Computing the continued fraction expansion

We implement the Euclidean algorithm to extract partial quotients and build convergents.

In [ ]:
def continued_fraction(x, n_terms=50):
    """Compute partial quotients [a0; a1, ..., a_{n-1}] of x."""
    a = []
    for _ in range(n_terms):
        a_k = int(np.floor(x))
        a.append(a_k)
        frac = x - a_k
        if frac < 1e-14:
            break
        x = 1.0 / frac
    return a


def convergents(a):
    """Compute convergents p_n/q_n from partial quotients."""
    p = [a[0], a[0]*a[1]+1 if len(a) > 1 else a[0]]
    q = [1,    a[1]        if len(a) > 1 else 1]
    for k in range(2, len(a)):
        p.append(a[k]*p[-1] + p[-2])
        q.append(a[k]*q[-1] + q[-2])
    return np.array(p[:len(a)]), np.array(q[:len(a)])


# Classic examples
specials = {
    r'$\pi$':             np.pi,
    r'$\sqrt{2}$':        np.sqrt(2),
    r'$\phi$ (golden)':   (1+np.sqrt(5))/2,
    r'$e$':               np.e,
    r'$2^{1/3}$':         2**(1/3),
}

for name, x in specials.items():
    a = continued_fraction(x, 15)
    print(f'{name} = [{a[0]}; {a[1:8]}...]')

## Convergents as best rational approximations

We plot the approximation error $|x - p_n/q_n|$ vs the denominator $q_n$ on a log-log scale. The $O(1/q^2)$ theoretical bound is shown. The golden ratio is the slowest converger; $\pi$ is faster due to some large partial quotients.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
cols = plt.cm.tab10(np.linspace(0, 0.5, len(specials)))

for (name, x), col in zip(specials.items(), cols):
    a = continued_fraction(x, 40)
    P, Q = convergents(a)
    errors = np.abs(P/Q - x)
    valid = (errors > 1e-16) & (Q < 1e15)
    ax.loglog(Q[valid], errors[valid], '.-', ms=8, lw=2, color=col, label=name)

# Theoretical bound O(1/q^2)
q_range = np.logspace(0, 10, 100)
ax.loglog(q_range, 1/q_range**2, 'k--', lw=1.5, label='$1/q^2$ bound')

ax.set_xlabel('Denominator $q_n$')
ax.set_ylabel(r'$|x - p_n/q_n|$')
ax.set_title('Convergents: best rational approximations')
ax.legend(fontsize=9); ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Partial quotient statistics

For a **generic** real number (Lebesgue almost every $x$), the partial quotients satisfy the **Gauss–Kuzmin distribution**: $\Pr[a_n = k] = -\log_2(1 - 1/(k+1)^2)$ for $k = 1, 2, \ldots$. Numbers with all $a_n = 1$ (like $\phi$) are exceptional and hardest to approximate.

In [ ]:
# Gauss-Kuzmin distribution
k_vals = np.arange(1, 20)
gk = -np.log2(1 - 1/(k_vals+1)**2)
gk /= gk.sum()  # normalize

# Empirical: random numbers
rng = np.random.default_rng(42)
n_rand = 500; n_terms = 100
all_quotients = []
for _ in range(n_rand):
    x = rng.uniform(0, 1)
    a = continued_fraction(x, n_terms)
    all_quotients.extend([ai for ai in a[1:] if 1 <= ai <= 20])  # skip a_0

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(all_quotients, bins=np.arange(0.5, 21.5), density=True,
             color='steelblue', alpha=0.7, label='empirical')
axes[0].plot(k_vals, gk, 'r-o', ms=6, lw=2, label='Gauss–Kuzmin')
axes[0].set_xlabel('partial quotient $a_k$'); axes[0].set_ylabel('frequency')
axes[0].set_title('Distribution of partial quotients'); axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Compare partial quotient sequences
for (name, x), col in zip(list(specials.items())[:4], plt.cm.tab10(np.linspace(0,0.4,4))):
    a = continued_fraction(x, 30)[1:25]  # skip a_0
    axes[1].plot(a, '.-', ms=7, lw=1.5, color=col, label=name)
axes[1].set_xlabel('$n$'); axes[1].set_ylabel('$a_n$')
axes[1].set_title('Partial quotient sequences'); axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Stern–Brocot tree and the Farey sequence

The **Stern–Brocot tree** organizes all positive rationals in a binary tree via the **mediant** operation: from $p/q$ and $p'/q'$, form $(p+p')/(q+q')$. This tree underlies the structure of continued fraction convergents.

In [ ]:
def stern_brocot_level(n_levels=5):
    """Generate Stern-Brocot tree nodes up to n_levels."""
    nodes = [(0, 1, 1, 1)]  # (left_p, left_q, right_p, right_q) for left/right bounds
    fracs = []
    queue = [(0, 1, 1, 1)]
    for _ in range(n_levels):
        next_queue = []
        for lp, lq, rp, rq in queue:
            mp, mq = lp+rp, lq+rq
            fracs.append((mp, mq))
            next_queue.append((lp, lq, mp, mq))
            next_queue.append((mp, mq, rp, rq))
        queue = next_queue
    return fracs

fracs = stern_brocot_level(6)
vals = [p/q for p, q in fracs]
denoms = [q for p, q in fracs]

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(vals, denoms, s=10, c='steelblue', alpha=0.5)
ax.set_xlabel('Rational value $p/q$'); ax.set_ylabel('Denominator $q$')
ax.set_title('Stern–Brocot tree: all fractions $p/q$ with $q \\leq 64$')
ax.set_xlim(0, 1); ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Interactive: explore a specific number

Choose a number and see its partial quotients, convergents, and how rapidly they approach the true value.

In [ ]:
def show_cf(number=r'$\pi$', n_terms=20):
    x = specials[number]
    a = continued_fraction(x, n_terms)
    P, Q = convergents(a)
    errors = np.abs(P/Q - x)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].stem(np.arange(len(a)), a, linefmt='b-', markerfmt='bo', basefmt='k')
    axes[0].set_xlabel('$n$'); axes[0].set_ylabel('$a_n$')
    axes[0].set_title(f'Partial quotients of {number}')
    axes[0].grid(alpha=0.3)

    valid = errors > 1e-16
    axes[1].loglog(Q[valid], errors[valid], 'b.-', ms=10, lw=2, label='convergents')
    q_range = np.logspace(0, np.log10(Q.max()+1), 100)
    axes[1].loglog(q_range, 1/q_range**2, 'k--', lw=1.5, label='$1/q^2$')
    axes[1].set_xlabel('$q_n$'); axes[1].set_ylabel(r'$|x - p_n/q_n|$')
    axes[1].set_title('Approximation error'); axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3, which='both')
    plt.tight_layout(); plt.show()

interact(show_cf,
         number=Dropdown(options=list(specials.keys()), description='number'),
         n_terms=IntSlider(value=20, min=5, max=45, step=5, description='terms'));

## Bibliographical resources

- Euler, L. (1748). *Introductio in analysin infinitorum*. Lausanne.
- Gauss, C. F. (1812). Disquisitiones generales circa seriem infinitam. *Commentationes Societatis Regiae Scientiarum Gottingensis*.
- Khinchin, A. Ya. (1964). *Continued Fractions*. University of Chicago Press.
- Hardy, G. H. and Wright, E. M. (2008). *An Introduction to the Theory of Numbers* (6th ed.). Oxford University Press. Chapter 10.
- Olds, C. D. (1963). *Continued Fractions*. Random House.